In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer as knn
from sklearn.preprocessing import KBinsDiscretizer
import plotly.express as px
from datetime import datetime


In [22]:
data = pd.read_csv(r'Data\data_to_train.csv')
data

,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Consumation,Engine,Power,Seats,Price,Brand,Model,car_age,km/age
0,Mumbai,2010,72000,CNG,Manual,First,26.60,998,58.16,5,1.75,Maruti,Wagon R LXI CNG,16,4500.000000
1,Pune,2015,41000,Diesel,Manual,First,19.67,1582,126.20,5,12.50,Hyundai,Creta 1.6 CRDi SX Option,11,3727.272727
2,Chennai,2011,46000,Petrol,Manual,First,18.20,1199,88.70,5,4.50,Honda,Jazz V,15,3066.666667
3,Chennai,2012,87000,Diesel,Manual,First,20.77,1248,88.76,7,6.00,Maruti,Ertiga VDI,14,6214.285714
4,Coimbatore,2013,40670,Diesel,Automatic,Second,15.20,1968,140.80,5,17.74,Audi,A4 New 2.0 TDI Multitronic,13,3128.461538
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5970,Delhi,2014,27365,Diesel,Manual,First,28.40,1248,74.00,5,4.75,Maruti,Swift VDI,12,2280.416667
5971,Jaipur,2015,100000,Diesel,Manual,First,24.40,1120,71.00,5,4.00,Hyundai,Xcent 1.1 CRDi S,11,9090.909091
5972,Jaipur,2012,55000,Diesel,Manual,Second,14.00,2498,112.00,8,2.90,Mahindra,Xylo D4 BSIV,14,3928.571429
5973,Kolkata,2013,46000,Petrol,Manual,First,18.90,998,67.10,5,2.65,Maruti,Wagon R VXI,13,3538.461538


In [23]:
data.describe()

,Year,Kilometers_Driven,Consumation,Engine,Power,Seats,Price,car_age,km/age
count,5975.000000,5.975000e+03,5975.000000,5975.000000,5975.000000,5975.000000,5975.000000,5975.000000,5975.000000
mean,2013.386778,5.867431e+04,18.179408,1621.606695,113.276894,5.278828,9.501647,12.613222,4615.731194
std,3.247238,9.155851e+04,4.521801,601.036987,53.415373,0.808959,11.205736,3.247238,9663.152695
min,1998.000000,1.710000e+02,0.000000,624.000000,34.200000,0.000000,0.440000,7.000000,24.428571
25%,2012.000000,3.390800e+04,15.200000,1198.000000,77.000000,5.000000,3.500000,10.000000,2909.090909
50%,2014.000000,5.300000e+04,18.160000,1493.000000,98.600000,5.000000,5.650000,12.000000,4200.000000
75%,2016.000000,7.300000e+04,21.100000,1984.000000,138.100000,5.000000,9.950000,14.000000,5533.666667
max,2019.000000,6.500000e+06,33.540000,5998.000000,560.000000,10.000000,160.000000,28.000000,722222.222222


In [24]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5975 entries, 0 to 5974
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Location           5975 non-null   str    
 1   Year               5975 non-null   int64  
 2   Kilometers_Driven  5975 non-null   int64  
 3   Fuel_Type          5975 non-null   str    
 4   Transmission       5975 non-null   str    
 5   Owner_Type         5975 non-null   str    
 6   Consumation        5975 non-null   float64
 7   Engine             5975 non-null   int64  
 8   Power              5975 non-null   float64
 9   Seats              5975 non-null   int64  
 10  Price              5975 non-null   float64
 11  Brand              5975 non-null   str    
 12  Model              5975 non-null   str    
 13  car_age            5975 non-null   int64  
 14  km/age             5975 non-null   float64
dtypes: float64(4), int64(5), str(6)
memory usage: 700.3 KB


In [25]:
data['Kilometers_Driven']

0        72000
1        41000
2        46000
3        87000
4        40670
         ...  
5970     27365
5971    100000
5972     55000
5973     46000
5974     47000
Name: Kilometers_Driven, Length: 5975, dtype: int64

# Dropping the data's that the km pass over the 1000000

In [7]:
index = data[data['Kilometers_Driven']>1000000].index
data.drop(index, inplace=True)

# doing the Normalization and capping

In [38]:
km_capping = np.clip(data[['Kilometers_Driven']], a_min=0, a_max=300000)
kbd= KBinsDiscretizer(n_bins=6,encode='ordinal', strategy='uniform')
data['KM_bin_encoded'] = kbd.fit_transform(km_capping).astype(int)
print(f"Bin edges used: {kbd.bin_edges_}")
print(data[['Kilometers_Driven', 'KM_bin_encoded']].tail())

Bin edges used: [array([1.710000e+02, 5.014250e+04, 1.001140e+05, 1.500855e+05,
        2.000570e+05, 2.500285e+05, 3.000000e+05])             ]
      Kilometers_Driven  KM_bin_encoded
5970              27365               0
5971             100000               1
5972              55000               1
5973              46000               0
5974              47000               0


In [39]:
data

,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Consumation,Engine,Power,Seats,Price,Brand,Model,car_age,km/age,KM_bin_encoded
0,Mumbai,2010,72000,CNG,Manual,First,26.60,998,58.16,5,1.75,Maruti,Wagon R LXI CNG,16,4500.000000,1
1,Pune,2015,41000,Diesel,Manual,First,19.67,1582,126.20,5,12.50,Hyundai,Creta 1.6 CRDi SX Option,11,3727.272727,0
2,Chennai,2011,46000,Petrol,Manual,First,18.20,1199,88.70,5,4.50,Honda,Jazz V,15,3066.666667,0
3,Chennai,2012,87000,Diesel,Manual,First,20.77,1248,88.76,7,6.00,Maruti,Ertiga VDI,14,6214.285714,1
4,Coimbatore,2013,40670,Diesel,Automatic,Second,15.20,1968,140.80,5,17.74,Audi,A4 New 2.0 TDI Multitronic,13,3128.461538,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5970,Delhi,2014,27365,Diesel,Manual,First,28.40,1248,74.00,5,4.75,Maruti,Swift VDI,12,2280.416667,0
5971,Jaipur,2015,100000,Diesel,Manual,First,24.40,1120,71.00,5,4.00,Hyundai,Xcent 1.1 CRDi S,11,9090.909091,1
5972,Jaipur,2012,55000,Diesel,Manual,Second,14.00,2498,112.00,8,2.90,Mahindra,Xylo D4 BSIV,14,3928.571429,1
5973,Kolkata,2013,46000,Petrol,Manual,First,18.90,998,67.10,5,2.65,Maruti,Wagon R VXI,13,3538.461538,0


In [40]:
px.histogram(data['KM_bin_encoded'])

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'bingroup': 'x',
              'hovertemplate': 'variable=KM_bin_encoded<br>value=%{x}<br>count=%{y}<extra></extra>',
              'legendgroup': 'KM_bin_encoded',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': 'KM_bin_encoded',
              'orientation': 'v',
              'showlegend': True,
              'type': 'histogram',
              'x': {'bdata': ('AQAAAQABAQABAQABAQEBAgEAAQEAAQ' ... 'EBAAEBAAEBAAIBAAAAAQABAAEBAAA='),
                    'dtype': 'i1'},
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'title': {'text': 'variable'}, 'tracegroupgap': 0},
               'margin': {'t': 60},
               'template': '...',
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'value'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'count'}}}
})

In [42]:
km_capping = np.clip(data[['Engine']], a_min=0, a_max=5500)
kbd = KBinsDiscretizer(n_bins=12, encode='ordinal',strategy='uniform')
data['Engine-bins'] = kbd.fit_transform(km_capping).astype(int)

In [43]:
data['Engine-bins'].value_counts()

Engine-bins
1     2212
2     1399
3     1015
0      587
5      405
4      296
6       19
7       14
8        8
9        8
10       7
11       5
Name: count, dtype: int64

In [45]:
px.histogram(data['Engine-bins'])

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'bingroup': 'x',
              'hovertemplate': 'variable=Engine-bins<br>value=%{x}<br>count=%{y}<extra></extra>',
              'legendgroup': 'Engine-bins',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': 'Engine-bins',
              'orientation': 'v',
              'showlegend': True,
              'type': 'histogram',
              'x': {'bdata': ('AAIBAQMAAgUCAQICAQMDBAIBAgMDAQ' ... 'ABAQEDAgABAQUBBQQBBwQCAQEEAAA='),
                    'dtype': 'i1'},
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'title': {'text': 'variable'}, 'tracegroupgap': 0},
               'margin': {'t': 60},
               'template': '...',
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'value'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'count'}}}
})

In [46]:
px.histogram(data['Engine'], nbins=20)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'bingroup': 'x',
              'hovertemplate': 'variable=Engine<br>value=%{x}<br>count=%{y}<extra></extra>',
              'legendgroup': 'Engine',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': 'Engine',
              'nbinsx': 20,
              'orientation': 'v',
              'showlegend': True,
              'type': 'histogram',
              'x': {'bdata': ('5gMuBq8E4ASwBy4DtQXDCj4G4AS2Bd' ... 'u+Ca4EDQ6+CdoF4ARgBMIJ5gOoAw=='),
                    'dtype': 'i2'},
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'title': {'text': 'variable'}, 'tracegroupgap': 0},
               'margin': {'t': 60},
               'template': '...',
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'value'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'count'}}}
})

In [47]:
power_capped = np.clip(data[['Power']], a_min=0, a_max=500)
kbd = KBinsDiscretizer(n_bins=10, encode='ordinal', strategy='uniform')

data['Power_bins'] = kbd.fit_transform(power_capped).astype(int)

In [48]:
px.histogram(data['Power_bins'])

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'bingroup': 'x',
              'hovertemplate': 'variable=Power_bins<br>value=%{x}<br>count=%{y}<extra></extra>',
              'legendgroup': 'Power_bins',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': 'Power_bins',
              'orientation': 'v',
              'showlegend': True,
              'type': 'histogram',
              'x': {'bdata': ('AAEBAQIAAAIBAAEBAAMBAwEAAQIDAQ' ... 'AAAQEDAQABAQQABQIBBAEBAAABAAA='),
                    'dtype': 'i1'},
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'title': {'text': 'variable'}, 'tracegroupgap': 0},
               'margin': {'t': 60},
               'template': '...',
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'value'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'count'}}}
})

In [49]:
data['Consumation'].value_counts()

Consumation
18.90    172
17.00    171
18.60    119
20.36     88
21.10     87
        ... 
30.46      1
6.40       1
12.85      1
18.69      1
17.24      1
Name: count, Length: 430, dtype: int64

In [ ]:
brand_counts = df['Brand'].value_counts()

# 2. Identify brands that appear less than 5 times
rare_brands = brand_counts[brand_counts < 5].index

# 3. Replace them in the DataFrame
df['Brand'] = df['Brand'].replace(rare_brands, 'Other')

# Check the results
print(f"Number of rare brands grouped: {len(rare_brands)}")
print(df['Brand'].value_counts().tail())

In [51]:
index =  data[data["Consumation"] < 5 ].index
data.drop(index,inplace = True)

In [52]:
data['Consumation'].min()

np.float64(6.4)

In [53]:
px.histogram(data['Consumation'])

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'bingroup': 'x',
              'hovertemplate': 'variable=Consumation<br>value=%{x}<br>count=%{y}<extra></extra>',
              'legendgroup': 'Consumation',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': 'Consumation',
              'orientation': 'v',
              'showlegend': True,
              'type': 'histogram',
              'x': {'bdata': ('mpmZmZmZOkDsUbgehaszQDMzMzMzMz' ... 'AAAAAALEBmZmZmZuYyQHE9CtejcDlA'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'title': {'text': 'variable'}, 'tracegroupgap': 0},
               'margin': {'t': 60},
               'template': '...',
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'value'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'count'}}}
})

In [54]:
px.histogram(data['Price'])

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'bingroup': 'x',
              'hovertemplate': 'variable=Price<br>value=%{x}<br>count=%{y}<extra></extra>',
              'legendgroup': 'Price',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': 'Price',
              'orientation': 'v',
              'showlegend': True,
              'type': 'histogram',
              'x': {'bdata': ('AAAAAAAA/D8AAAAAAAApQAAAAAAAAB' ... 'MzMzMzB0AzMzMzMzMFQAAAAAAAAARA'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'title': {'text': 'variable'}, 'tracegroupgap': 0},
               'margin': {'t': 60},
               'template': '...',
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'value'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'count'}}}
})

In [55]:
brand_counts = data['Brand'].value_counts()
rare_brands = brand_counts[brand_counts < 5].index

data['Brand'] = data['Brand'].replace(rare_brands, 'Other')

print(f"Number of rare brands grouped: {len(rare_brands)}")
print(data['Brand'].value_counts().tail())

Number of rare brands grouped: 6
Brand
Volvo      21
Porsche    18
Jeep       15
Datsun     13
Other       9
Name: count, dtype: int64


In [56]:
data['Power_to_Engine_Ratio'] = data['Power'] / (data['Engine'] + 1e-6)

In [58]:
data.columns

Index(['Location', 'Year', 'Kilometers_Driven', 'Fuel_Type', 'Transmission',
       'Owner_Type', 'Consumation', 'Engine', 'Power', 'Seats', 'Price',
       'Brand', 'Model', 'car_age', 'km/age', 'KM_bin_encoded', 'Engine-bins',
       'Power_bins', 'Power_to_Engine_Ratio'],
      dtype='str')

In [59]:
data['Mileage_per_Year'] = data['Kilometers_Driven'] / (data['car_age'] + 1)

In [60]:
# Mileage efficiency (fuel efficiency weighted by power)
data['Mileage_Efficiency'] = data['Consumation'] * data['Power'] / (data['Engine'] + 1e-6)

In [62]:
# Depreciation factor (older cars with more km depreciate more)
data['Depreciation_Factor'] = data['car_age'] * data['Kilometers_Driven'] / 100000

In [64]:
# Value retention (newer cars with less km retain value better)
current_year = datetime.now().year
data['Value_Retention'] = (current_year - data['Year']) / (data['Kilometers_Driven'] / 1000 + 1)

In [65]:
# Performance index (power and engine combination)
data['Performance_Index'] = (data['Power'] * data['Engine']) / 1000

In [67]:
# Economy index (mileage and efficiency)
data['Economy_Index'] = data['Consumation'] * (1 / (data['Engine'] / 1000 + 1))

# Car_Age: Age of the car (2024 - Year)
# Power_to_Engine_Ratio: Efficiency metric (Power/Engine)
# Mileage_per_Year: Usage intensity (Kilometers_Driven / Car_Age)
# Engine_per_Seat: Comfort metric (Engine capacity per seat)
# Power_per_Seat: Power per seat
# Mileage_Efficiency: Fuel efficiency weighted by power
# Depreciation_Factor: Age × Kilometers (depreciation indicator)
# Value_Retention: Newer cars with less km retain value better
# Performance_Index: Power × Engine combination
# Economy_Index: Mileage efficiency metric

In [68]:
data.info()

<class 'pandas.DataFrame'>
Index: 5919 entries, 0 to 5974
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Location               5919 non-null   str    
 1   Year                   5919 non-null   int64  
 2   Kilometers_Driven      5919 non-null   int64  
 3   Fuel_Type              5919 non-null   str    
 4   Transmission           5919 non-null   str    
 5   Owner_Type             5919 non-null   str    
 6   Consumation            5919 non-null   float64
 7   Engine                 5919 non-null   int64  
 8   Power                  5919 non-null   float64
 9   Seats                  5919 non-null   int64  
 10  Price                  5919 non-null   float64
 11  Brand                  5919 non-null   str    
 12  Model                  5919 non-null   str    
 13  car_age                5919 non-null   int64  
 14  km/age                 5919 non-null   float64
 15  KM_bin_encoded      

In [69]:
data.describe()

,Year,Kilometers_Driven,Consumation,Engine,Power,Seats,Price,car_age,km/age,KM_bin_encoded,Engine-bins,Power_bins,Power_to_Engine_Ratio,Mileage_per_Year,Mileage_Efficiency,Depreciation_Factor,Value_Retention,Performance_Index,Economy_Index
count,5919.000000,5.919000e+03,5919.000000,5919.000000,5919.000000,5919.000000,5919.000000,5919.000000,5919.000000,5919.000000,5919.000000,5919.000000,5919.000000,5919.000000,5919.000000,5919.000000,5919.000000,5919.000000,5919.000000
mean,2013.424396,5.852937e+04,18.351404,1620.266092,113.141351,5.280284,9.478713,12.575604,4618.769384,0.642507,1.960973,1.192769,0.068945,4264.295926,1.256872,7.860787,0.315847,210.899648,7.524067
std,3.207900,9.186271e+04,4.181298,598.449738,53.470138,0.808038,11.157823,3.207900,9706.105779,0.715486,1.445997,1.184633,0.013137,8754.035140,0.325975,9.809595,0.360469,205.246372,2.726384
min,1998.000000,1.710000e+02,6.400000,624.000000,34.200000,0.000000,0.440000,7.000000,24.428571,0.000000,0.000000,0.000000,0.023842,21.375000,0.296709,0.011970,0.001384,21.840000,1.031593
25%,2012.000000,3.385650e+04,15.300000,1198.000000,76.800000,5.000000,3.500000,10.000000,2909.090909,0.000000,1.000000,0.000000,0.061635,2666.884615,1.027994,3.600000,0.178131,95.132000,5.348211
50%,2014.000000,5.300000e+04,18.200000,1493.000000,98.600000,5.000000,5.650000,12.000000,4205.823529,1.000000,2.000000,1.000000,0.067234,3888.888889,1.294737,6.490000,0.232621,141.369563,7.734304
75%,2016.000000,7.269450e+04,21.100000,1984.000000,138.100000,5.000000,9.935000,14.000000,5536.230769,1.000000,3.000000,2.000000,0.076241,5123.650000,1.460890,10.120700,0.333333,278.534000,9.267183
max,2019.000000,6.500000e+06,33.540000,5998.000000,560.000000,10.000000,160.000000,28.000000,722222.222222,5.000000,11.000000,9.000000,0.153421,650000.000000,2.608278,585.000000,8.403361,3310.896000,18.619154


In [71]:
# Drop columns that are no longer needed (Location is kept for Target Encoding)
data.drop(["Kilometers_Driven","Engine","Power"],axis = 1,inplace = True)

In [72]:
data

,Location,Year,Fuel_Type,Transmission,Owner_Type,Consumation,Seats,Price,Brand,Model,...,KM_bin_encoded,Engine-bins,Power_bins,Power_to_Engine_Ratio,Mileage_per_Year,Mileage_Efficiency,Depreciation_Factor,Value_Retention,Performance_Index,Economy_Index
0,Mumbai,2010,CNG,Manual,First,26.60,5,1.75,Maruti,Wagon R LXI CNG,...,1,0,0,0.058277,4235.294118,1.550156,11.5200,0.219178,58.04368,13.313313
1,Pune,2015,Diesel,Manual,First,19.67,5,12.50,Hyundai,Creta 1.6 CRDi SX Option,...,0,2,1,0.079772,3416.666667,1.569124,4.5100,0.261905,199.64840,7.618125
2,Chennai,2011,Petrol,Manual,First,18.20,5,4.50,Honda,Jazz V,...,0,1,1,0.073978,2875.000000,1.346405,6.9000,0.319149,106.35130,8.276489
3,Chennai,2012,Diesel,Manual,First,20.77,7,6.00,Maruti,Ertiga VDI,...,1,1,1,0.071122,5800.000000,1.477200,12.1800,0.159091,110.77248,9.239324
4,Coimbatore,2013,Diesel,Automatic,Second,15.20,5,17.74,Audi,A4 New 2.0 TDI Multitronic,...,0,3,2,0.071545,2905.000000,1.087480,5.2871,0.311975,277.09440,5.121294
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5970,Delhi,2014,Diesel,Manual,First,28.40,5,4.75,Maruti,Swift VDI,...,0,1,0,0.059295,2105.000000,1.683974,3.2838,0.423057,92.35200,12.633452
5971,Jaipur,2015,Diesel,Manual,First,24.40,5,4.00,Hyundai,Xcent 1.1 CRDi S,...,1,1,0,0.063393,8333.333333,1.546786,11.0000,0.108911,79.52000,11.509434
5972,Jaipur,2012,Diesel,Manual,Second,14.00,8,2.90,Mahindra,Xylo D4 BSIV,...,1,4,1,0.044836,3666.666667,0.627702,7.7000,0.250000,279.77600,4.002287
5973,Kolkata,2013,Petrol,Manual,First,18.90,5,2.65,Maruti,Wagon R VXI,...,0,0,0,0.067234,3285.714286,1.270731,5.9800,0.276596,66.96580,9.459459
